### 570. Managers with at Least 5 Direct Reports
```

Table: Employee

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
| department  | varchar |
| managerId   | int     |
+-------------+---------+
id is the primary key (column with unique values) for this table.
Each row of this table indicates the name of an employee, their department, and the id of their manager.
If managerId is null, then the employee does not have a manager.
No employee will be the manager of themself.
 

Write a solution to find managers with at least five direct reports.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Employee table:
+-----+-------+------------+-----------+
| id  | name  | department | managerId |
+-----+-------+------------+-----------+
| 101 | John  | A          | null      |
| 102 | Dan   | A          | 101       |
| 103 | James | A          | 101       |
| 104 | Amy   | A          | 101       |
| 105 | Anne  | A          | 101       |
| 106 | Ron   | B          | 101       |
+-----+-------+------------+-----------+
Output: 
+------+
| name |
+------+
| John |
+------+

```

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col,count

In [5]:
spark=SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [6]:
Data = [[101, 'John', 'A', None], [102, 'Dan', 'A', 101], [103, 'James', 'A', 101], [104, 'Amy', 'A', 101], [105, 'Anne', 'A', 101], [106, 'Ron', 'B', 101]]

Schema=StructType([StructField("id",IntegerType(),True),\
                  StructField("name",StringType(),True),\
                  StructField("department",StringType(),True),\
                  StructField("managerId",IntegerType(),True)]
                  
                 )
Employee=spark.createDataFrame(Data,Schema)
Employee.show()

+---+-----+----------+---------+
| id| name|department|managerId|
+---+-----+----------+---------+
|101| John|         A|     NULL|
|102|  Dan|         A|      101|
|103|James|         A|      101|
|104|  Amy|         A|      101|
|105| Anne|         A|      101|
|106|  Ron|         B|      101|
+---+-----+----------+---------+



In [13]:
result = (Employee.alias("E").
    join(Employee.alias("M"), 
         col("E.managerId") == col("M.id"),
         "inner")
    .groupBy(col("M.name"))
    .agg(count("E.id").alias("emp_count"))
    .filter(col("emp_count")>=5)
    .select(col("m.name"))
         )

In [14]:
result.show()

+----+
|name|
+----+
|John|
+----+



In [15]:
spark.stop()